# EDA — LBNL Simulated RTU Dataset: Refrigerant Overcharge Fault

**Goal**: refrigerant overcharge is the physical opposite of undercharge
(too much refrigerant, not too little) - same sensors, same system,
opposite direction of fault. Hypothesis to test: does overcharge produce
the OPPOSITE pattern from what we found for undercharge (lower suction
pressure/temp, higher discharge pressure, and - the interesting
question - does capacity also degrade, or does it behave differently
since this isn't a starvation problem?)

See ml/notebooks/01_eda_undercharge.ipynb for the data dictionary and the
segmented-EWMA feature-engineering rule - not repeating those here.

In [1]:
import pandas as pd

files = {
    "baseline": "../data/raw/RTU_sim_baseline.csv",
    "overcharge10": "../data/raw/RTU_sim_overcharge10.csv",
    "overcharge15": "../data/raw/RTU_sim_overcharge15.csv",
    "overcharge20": "../data/raw/RTU_sim_overcharge20.csv",
}

dfs = {label: pd.read_csv(fname) for label, fname in files.items()}

for label, df in dfs.items():
    print(f"{label}: shape={df.shape}, missing_values={df.isna().sum().sum()}")

baseline: shape=(143941, 25), missing_values=0
overcharge10: shape=(143941, 25), missing_values=0
overcharge15: shape=(143941, 25), missing_values=0
overcharge20: shape=(143941, 25), missing_values=0


In [2]:
import sys
from pathlib import Path

# ml/ is not an installed package (see ml/pyproject.toml: package-mode = false —
# deliberate, since ml/ isn't a deployed service). Notebooks live in ml/notebooks/,
# so add the ml/ root to sys.path to make `from src.features...` imports work
# consistently, the same way pytest's pythonpath = ["."] setting already does
# for the test suite.
ml_root = Path.cwd().parent
if str(ml_root) not in sys.path:
    sys.path.insert(0, str(ml_root))

from src.features.effect_size import cohens_d  # noqa: E402
from src.features.smoothing import add_segmented_ewma  # noqa: E402

In [3]:
key_cols = ["RTU_REFG_SUCT_PRES", "RTU_REFG_DISC_PRES", "RTU_REFG_SUCT_TEMP", "RTU_TOT_CAPA"]

summary = pd.DataFrame({
    label: dfs[label][dfs[label]["RTU_STG_STA"] >= 0.6][key_cols].mean()
    for label in files
})
print(summary)

                        baseline  overcharge10  overcharge15  overcharge20
RTU_REFG_SUCT_PRES  1.473838e+07  1.471943e+07  1.460175e+07  1.456556e+07
RTU_REFG_DISC_PRES  2.597149e+07  2.464363e+07  2.608237e+07  2.573726e+07
RTU_REFG_SUCT_TEMP  5.364631e+01  5.344421e+01  5.305362e+01  5.283428e+01
RTU_TOT_CAPA        1.426396e+04  1.432419e+04  1.453598e+04  1.390073e+04


## Result — overcharge is NOT a clean mirror image of undercharge

Suction pressure and suction temperature both move steadily in one
direction as severity increases (mild decrease, 10% -> 20%) - consistent
with a real fault signature, similar in character (if smaller in
magnitude) to undercharge's suction-side behavior.

But discharge pressure and total capacity are **non-monotonic**:
discharge pressure drops at 10%, jumps back up at 15% (even above
baseline), then drops again at 20%. Capacity actually *increases*
slightly at 10% and 15% before dropping at 20%.

**This means overcharge's effect on the refrigeration cycle is more
complex than undercharge's** - it doesn't cleanly worsen a single
metric as severity increases. A classifier trained on simple linear
severity trends (like we might build for undercharge) would NOT
transfer to overcharge without accounting for this non-monotonic
behavior. Worth flagging as a real modeling consideration, not
something to smooth over or explain away without more investigation.

## 2b — checking the known confound: is compressor staging behind the non-monotonic discharge/capacity pattern?

We already found once today that RTU_STG_STA blends genuinely different
operating regimes (stage 1 vs stage 2) into one comparison if not
handled carefully. Before treating overcharge's non-monotonic pattern as
a new, unexplained phenomenon, check whether the SAME confound explains
it - i.e., does the proportion of stage-1 vs stage-2 running time differ
meaningfully across these four files? If baseline happened to run more
in stage 2 than overcharge15 did, that alone could produce exactly this
kind of jumbled comparison.

In [4]:
for label in files:
    subset = dfs[label][dfs[label]["RTU_STG_STA"] >= 0.6]
    stage2_fraction = (subset["RTU_STG_STA"] >= 0.9).mean()
    print(f"{label}: fraction of active-running time in stage 2 = {stage2_fraction:.3f}")

baseline: fraction of active-running time in stage 2 = 0.557
overcharge10: fraction of active-running time in stage 2 = 0.553
overcharge15: fraction of active-running time in stage 2 = 0.554
overcharge20: fraction of active-running time in stage 2 = 0.436


## 2c — does restricting to pure stage-2 operation remove the non-monotonicity?

Stage-2 fraction only explains part of the picture (it doesn't account
for the 10% -> 15% jump, since those two files have nearly identical
stage-2 fractions). Tightening the filter to RTU_STG_STA >= 0.9 (pure
stage 2 only, no stage-1 minutes mixed in) removes even the partial
blending the looser >= 0.6 filter still allowed.

In [5]:
summary_stage2_only = pd.DataFrame({
    label: dfs[label][dfs[label]["RTU_STG_STA"] >= 0.9][key_cols].mean()
    for label in files
})
print(summary_stage2_only)

                        baseline  overcharge10  overcharge15  overcharge20
RTU_REFG_SUCT_PRES  1.435190e+07  1.432275e+07  1.421263e+07  1.413987e+07
RTU_REFG_DISC_PRES  2.706341e+07  2.547235e+07  2.729915e+07  2.733602e+07
RTU_REFG_SUCT_TEMP  5.329764e+01  5.312358e+01  5.267750e+01  5.239796e+01
RTU_TOT_CAPA        1.630269e+04  1.642002e+04  1.662485e+04  1.655100e+04


## Result — the confound explains part of the anomaly, but not all of it

Tightening to strict stage-2 operation (RTU_STG_STA >= 0.9) mostly
resolved RTU_TOT_CAPA's non-monotonicity - the 20% drop shrank from
~4.4% (loose filter) to under 0.5% (strict filter), consistent with the
compressor-staging confound being the main cause there.

RTU_REFG_DISC_PRES remains genuinely non-monotonic even under strict
filtering: it drops at 10% severity, then rises back ABOVE baseline at
15% and 20%. This is not explained by the staging confound we already
know about - it appears to be a real feature of how overcharge affects
discharge pressure, not an artifact of our analysis.

**Honest conclusion, not fully resolved**: overcharge's effect on
discharge pressure is more complex than a simple "moves consistently
with severity" relationship - possibly because overcharge causes
different behavior in the condenser (excess liquid backing up, changing
subcooling) that doesn't scale linearly with how much extra refrigerant
is present. Fully explaining the underlying refrigeration-cycle physics
here would need deeper domain expertise than we have from data alone -
documenting as a genuine, open finding rather than guessing further.

**What we can rely on**: suction pressure and suction temperature both
remain cleanly monotonic across all severities, even under strict
filtering - these are the trustworthy signals for an overcharge
classifier. Discharge pressure and capacity need non-linear treatment
(e.g. absolute deviation from baseline, not signed direction) if used
at all.

##  3 — does our segmented-EWMA feature generalize to overcharge?

We built add_segmented_ewma() based on undercharge data. Before assuming
it's a genuinely reusable tool (not just something that happened to work
once), check it against overcharge's suction temperature - one of the
two signals here that behaves cleanly (monotonic even under strict
filtering), unlike discharge pressure.

In [6]:
for label, df_ in dfs.items():
    dfs[label] = add_segmented_ewma(df_, value_col="RTU_REFG_SUCT_TEMP", state_col="RTU_STG_STA", span=30)

stage2_baseline = dfs["baseline"][dfs["baseline"]["RTU_STG_STA"] >= 0.9]
stage2_oc20 = dfs["overcharge20"][dfs["overcharge20"]["RTU_STG_STA"] >= 0.9]

raw_shift = stage2_baseline["RTU_REFG_SUCT_TEMP"].mean() - stage2_oc20["RTU_REFG_SUCT_TEMP"].mean()
raw_std = stage2_baseline["RTU_REFG_SUCT_TEMP"].std()

ewma_col = "RTU_REFG_SUCT_TEMP_ewma30_segmented"
ewma_shift = stage2_baseline[ewma_col].mean() - stage2_oc20[ewma_col].mean()
ewma_std = stage2_baseline[ewma_col].std()

print(f"RAW  -> shift: {raw_shift:.3f}, std: {raw_std:.3f}, effect size: {raw_shift/raw_std:.2f}")
print(f"EWMA -> shift: {ewma_shift:.3f}, std: {ewma_std:.3f}, effect size: {ewma_shift/ewma_std:.2f}")

RAW  -> shift: 0.900, std: 2.754, effect size: 0.33
EWMA -> shift: 0.634, std: 2.154, effect size: 0.29


## Result — segmented EWMA did NOT generalize cleanly to this signal

Unlike undercharge's RTU_TOT_CAPA (where segmented EWMA improved effect
size from 1.44 to 1.59), overcharge's RTU_REFG_SUCT_TEMP showed no
improvement (0.33 raw -> 0.29 EWMA, slightly worse).

**Two honest possible explanations, not yet fully verified:**
1. The raw signal here is already weak (effect size 0.33 vs undercharge
   capacity's 1.44) - there may simply be less real signal for smoothing
   to recover, unlike undercharge where a strong signal was being
   obscured by cross-regime noise.
2. If many stage-2 "runs" in this data are shorter than the EWMA span
   (30 minutes), most smoothed values within a short run never get much
   averaging history before the run ends - `add_segmented_ewma()` may
   behave differently on data with mostly-short segments than
   mostly-long ones like undercharge had.

**Not resolved tonight** - flagging as an open item for the actual
feature-engineering phase, where we'd check segment-length distributions
properly across all fault types before assuming one smoothing approach
works uniformly everywhere.

##  3b — checking hypothesis 2: are stage-2 "runs" shorter here than in undercharge?

If most contiguous stage-2 runs are shorter than the EWMA span (30
minutes), the segmented EWMA barely gets to smooth anything before each
run ends - which would explain why it didn't help here. Checking the
actual run-length distribution directly instead of guessing.

In [7]:
def get_run_lengths(df_, state_col="RTU_STG_STA"):
    stage_bucket = pd.cut(df_[state_col], bins=[-0.01, 0.3, 0.9, 1.01], labels=["off", "stage1", "stage2"])
    run_id = (stage_bucket != stage_bucket.shift()).cumsum()
    run_lengths = df_.groupby(run_id).size()
    # only keep runs that were actually stage2
    is_stage2 = df_.groupby(run_id)[state_col].first() >= 0.9
    return run_lengths[is_stage2]

for label in ["baseline", "overcharge20"]:
    lengths = get_run_lengths(dfs[label])
    print(f"{label} stage-2 run lengths (minutes) -> median: {lengths.median()}, mean: {lengths.mean():.1f}, count of runs under 30min: {(lengths < 30).sum()} / {len(lengths)}")

baseline stage-2 run lengths (minutes) -> median: 26.0, mean: 26.2, count of runs under 30min: 2255 / 2408
overcharge20 stage-2 run lengths (minutes) -> median: 16.0, mean: 16.1, count of runs under 30min: 2423 / 2423


## Result — hypothesis 2 confirmed: span=30 is too long for the actual run lengths

Baseline stage-2 runs: median 26 minutes, 93.6% (2255/2408) already
under our 30-minute EWMA span. Overcharge20 runs are even shorter:
median 16 minutes, **100%** under 30 minutes.

This means most stage-2 runs end before EWMA(span=30) ever accumulates
much real smoothing - the exponential weighting hasn't had time to
"forget" the transition-in noise before the run is already over. Our
span=30 choice was carried over from undercharge without checking
whether it actually fit this fault's dynamics - a real gap in our
earlier work, not something wrong with the segmented-EWMA approach
itself.

**Even undercharge's earlier "success" may have been partly marginal**
for the same reason (its baseline runs are also mostly under 30 min) -
worth revisiting once we build the real pipeline, not just assuming
span=30 is a safe universal default.

In [8]:
for label, df_ in dfs.items():
    dfs[label] = add_segmented_ewma(
        df_, value_col="RTU_REFG_SUCT_TEMP", state_col="RTU_STG_STA", span=10,
        output_col="RTU_REFG_SUCT_TEMP_ewma10_segmented"
    )

stage2_baseline = dfs["baseline"][dfs["baseline"]["RTU_STG_STA"] >= 0.9]
stage2_oc20 = dfs["overcharge20"][dfs["overcharge20"]["RTU_STG_STA"] >= 0.9]

ewma10_col = "RTU_REFG_SUCT_TEMP_ewma10_segmented"
shift_10 = stage2_baseline[ewma10_col].mean() - stage2_oc20[ewma10_col].mean()
std_10 = stage2_baseline[ewma10_col].std()

print("RAW        -> effect size: 0.33")
print("EWMA(30)   -> effect size: 0.29")
print(f"EWMA(10)   -> shift: {shift_10:.3f}, std: {std_10:.3f}, effect size: {shift_10/std_10:.2f}")

RAW        -> effect size: 0.33
EWMA(30)   -> effect size: 0.29
EWMA(10)   -> shift: 0.586, std: 2.340, effect size: 0.25


## Result — the span-mismatch hypothesis was wrong (or incomplete)

Prediction was that a shorter span (better matched to the ~16-26 minute
run lengths we measured) would recover EWMA's benefit. It didn't:
raw (0.33) -> EWMA span=30 (0.29) -> EWMA span=10 (0.25) - smoothing
gets steadily WORSE as it's applied, regardless of span tested.

**Revised, more honest explanation**: smoothing can only recover a real
signal that's being obscured by noise - it cannot manufacture separation
that isn't there. RTU_REFG_SUCT_TEMP's raw effect size for overcharge
(0.33) is already "small" by conventional standards, unlike undercharge's
RTU_TOT_CAPA (1.44, "large"). There may simply be much less real
fault-signal in this feature for this fault type - smoothing has
nothing strong to recover, and any smoothing at all only adds a small
amount of its own distortion (blurring real short-timescale variation
along with noise).

**Real lesson, not a failure**: segmented EWMA is not a universal fix -
it helps when a real signal is present but obscured by cross-regime
noise (undercharge/capacity), and does nothing (or mildly hurts) when
the underlying signal is already weak (overcharge/suction-temp). Span
tuning is a genuine hyperparameter search problem for the actual
feature-engineering/model-validation phase - not something to resolve
by hand-tuning against one severity comparison in EDA. Stopping the
span search here rather than chasing it further without a proper
validation setup.

**Practical implication for overcharge**: suction temperature alone,
smoothed or not, looks like a weak individual signal. The model will
likely need other features (discharge pressure, even non-monotonic, may
still carry information in combination with others) or a multivariate
approach - not a single strong univariate signal the way undercharge's
capacity was.

## Summary — overcharge EDA

1. **Not a mirror image of undercharge.** Suction pressure and suction
   temperature move cleanly, monotonically with severity (mild
   decrease) - but discharge pressure is genuinely non-monotonic even
   under strict stage-2 filtering, and total capacity's apparent
   non-monotonicity was mostly (not entirely) explained by the
   compressor-staging confound.
2. **Segmented EWMA is not a universal fix.** It clearly helped
   undercharge's capacity signal (weak-but-real signal, obscured by
   cross-regime noise). It did not help overcharge's suction
   temperature at any span tested (30 or 10) - likely because the raw
   signal there is already weak, and smoothing has nothing strong to
   recover.
3. **Open items for the real feature-engineering phase**, not resolved
   here:
   - Why does discharge pressure behave non-monotonically for
     overcharge? (Domain-expertise question, not resolvable from data
     alone in EDA.)
   - EWMA span should likely be validated per-feature/per-fault via
     proper hyperparameter search, not assumed from one manual check.
   - Overcharge likely needs multiple features combined (not one strong
     univariate signal like undercharge had) - a hint that our eventual
     classifier will need real feature selection/combination, not just
     "pick the biggest effect size."

In [9]:
d_raw = cohens_d(
    dfs["baseline"][dfs["baseline"]["RTU_STG_STA"] >= 0.9]["RTU_REFG_SUCT_TEMP"],
    dfs["overcharge20"][dfs["overcharge20"]["RTU_STG_STA"] >= 0.9]["RTU_REFG_SUCT_TEMP"],
)
d_ewma30 = cohens_d(
    dfs["baseline"][dfs["baseline"]["RTU_STG_STA"] >= 0.9]["RTU_REFG_SUCT_TEMP_ewma30_segmented"],
    dfs["overcharge20"][dfs["overcharge20"]["RTU_STG_STA"] >= 0.9]["RTU_REFG_SUCT_TEMP_ewma30_segmented"],
)
d_ewma10 = cohens_d(
    dfs["baseline"][dfs["baseline"]["RTU_STG_STA"] >= 0.9]["RTU_REFG_SUCT_TEMP_ewma10_segmented"],
    dfs["overcharge20"][dfs["overcharge20"]["RTU_STG_STA"] >= 0.9]["RTU_REFG_SUCT_TEMP_ewma10_segmented"],
)

print("Baseline-std method (as computed above): raw=0.33, ewma30=0.29, ewma10=0.25")
print(f"Pooled-std method (cohens_d):             raw={d_raw:.2f}, ewma30={d_ewma30:.2f}, ewma10={d_ewma10:.2f}")

Baseline-std method (as computed above): raw=0.33, ewma30=0.29, ewma10=0.25
Pooled-std method (cohens_d):             raw=0.32, ewma30=0.28, ewma10=0.24


## Result — pooled-std confirms the same conclusion

| Method | Raw | EWMA(30) | EWMA(10) |
|---|---|---|---|
| Baseline-std (as computed above) | 0.33 | 0.29 | 0.25 |
| Pooled-std (`cohens_d()`) | 0.32 | 0.28 | 0.24 |

Consistent, small (~0.01) downward shift under pooled-std, same as notebook 01's
reconciliation — no change to the conclusion. All three effect sizes remain solidly
"small" under either convention, and the ranking (raw > EWMA30 > EWMA10, smoothing
making things progressively worse) is unaffected by the choice of method.